# Dataset BIO Generation and Reporting

This notebook has two main stages:

1. Generate BIO `.txt` datasets from the curated NER/LangExtract alignment outputs. The first cells read the curated JSON/JSONL samples, keep the exact-match or curated training candidates, convert their entities into BIO token/tag lines, and save the resulting training candidate files under `resources/data/restricted/training-datasets`.

2. Report and compare the BIO datasets. The reporting section parses the BIO files, compares J10 old vs new splits, analyzes each candidate dataset individually, checks which labels are present or absent compared with the base train label set, and evaluates which external dataset combinations satisfy the required label coverage for future Flair NER fine-tuning.


### Available Datasets

#### - Juz 10 (original reviewed + new samples)
#### - Consejo de la Magistratura
#### - SAIJ
#### - PJ Jujuy
#### - PII-400
#### - Defensoría del Pueblo
#### - Data Augmentation
#### - Synthetic Paragraphs

In [ ]:
import json
from pathlib import Path
from typing import Any

from tqdm import tqdm


def read_samples(input_json_path: str | Path) -> list[dict[str, Any]]:
    """Read inference samples from a JSON array, JSONL file, or {'samples': [...]} payload."""
    input_json_path = Path(input_json_path)
    if not input_json_path.is_file():
        raise FileNotFoundError(f"Input JSON file not found: {input_json_path}")

    if input_json_path.suffix == ".jsonl":
        rows = []
        with input_json_path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSONL at {input_json_path}:{line_no}") from exc
        return rows

    with input_json_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("samples"), list):
        return payload["samples"]
    raise ValueError("Expected a JSON array, JSONL file, or object with a 'samples' list")


def get_exact_match_train_candidates(samples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Keep samples where NER and LangExtract agreed exactly."""
    return [
        sample
        for sample in samples
        if (sample.get("comparison") or {}).get("status") == "exact_match"
    ]


def get_train_candidate_entities(sample: dict[str, Any]) -> list[dict[str, Any]]:
    """Return entities confirmed by the exact-match comparison.

    Exact-match samples with no entities return [] and are written as all-O BIO.
    Curated samples can override the comparison entities with final_entities.
    """
    if isinstance(sample.get("final_entities"), list):
        return sample["final_entities"]

    comparison = sample.get("comparison") or {}
    if isinstance(comparison.get("exact_match"), list):
        return comparison["exact_match"]

    return []


def build_token_offsets(text: str, tokens: list[str]) -> list[tuple[int, int]]:
    offsets = []
    cursor = 0
    for token in tokens:
        start = text.find(token, cursor)
        if start == -1:
            raise ValueError(f"Could not align token {token!r} after char {cursor}")
        end = start + len(token)
        offsets.append((start, end))
        cursor = end
    return offsets


def entities_to_bio_lines(text: str, entities: list[dict[str, Any]]) -> list[str]:
    tokens = text.split()
    offsets = build_token_offsets(text, tokens)
    bio_tags = ["O"] * len(tokens)

    for entity in sorted(entities, key=lambda item: (item["start_char"], item["end_char"])):
        first = True
        for idx, (tok_start, tok_end) in enumerate(offsets):
            if entity["end_char"] <= tok_start or entity["start_char"] >= tok_end:
                continue
            prefix = "B" if first else "I"
            bio_tags[idx] = f"{prefix}-{entity['label']}"
            first = False

    return [f"{token} {tag}" for token, tag in zip(tokens, bio_tags)]


def sample_to_bio_lines(sample: dict[str, Any]) -> list[str]:
    text = sample.get("text") or ""
    entities = get_train_candidate_entities(sample)
    return entities_to_bio_lines(text, entities)


def generate_bio_from_train_candidates(
    input_json_path: str | Path,
    output_bio_txt_path: str | Path,
) -> list[dict[str, Any]]:
    """Filter exact-match inference samples and write them as BIO txt."""
    samples = read_samples(input_json_path)
    train_candidates = get_exact_match_train_candidates(samples)

    paragraphs = []
    skipped_sample_ids = []
    for sample in tqdm(train_candidates, desc="Converting exact matches to BIO"):
        try:
            bio_lines = sample_to_bio_lines(sample)
        except (KeyError, ValueError) as exc:
            skipped_sample_ids.append(sample.get("sample_id"))
            print(f"Skipping sample {sample.get('sample_id')}: {exc}")
            continue

        if bio_lines:
            paragraphs.append("\n".join(bio_lines))
        else:
            skipped_sample_ids.append(sample.get("sample_id"))

    output_bio_txt_path = Path(output_bio_txt_path)
    output_bio_txt_path.parent.mkdir(parents=True, exist_ok=True)
    output_text = "\n\n".join(paragraphs) + ("\n" if paragraphs else "")
    output_bio_txt_path.write_text(output_text, encoding="utf-8")

    print(f"Loaded samples: {len(samples)}")
    print(f"Train candidates with exact_match status: {len(train_candidates)}")
    print(f"BIO paragraphs written: {len(paragraphs)}")
    print(f"Skipped paragraphs: {len(skipped_sample_ids)}")
    print(f"Saved BIO txt: {output_bio_txt_path}")

    return train_candidates


In [ ]:
j10_raw_samples = '../../../resources/data/restricted/ner-langextract-alignment/curated/predictions_raw/restricted_samples_raw_curated.json'
j10_bio = '../../../resources/data/restricted/training-datasets/private/j10_new_train_candidates.txt'

pj_jujuy_raw_samples = '../../../resources/data/restricted/ner-langextract-alignment/curated/predictions_raw/samples_raw_jujuy.json'
pj_jujuy_bio = '../../../resources/data/restricted/training-datasets/private/pj_jujuy_train_candidates.txt'

saij_raw_samples = '../../../resources/data/restricted/ner-langextract-alignment/curated/predictions_raw/samples_raw_saij.json'
saij_bio = '../../../resources/data/restricted/training-datasets/private/saij_train_candidates.txt'

cm_raw_samples = '../../../resources/data/restricted/ner-langextract-alignment/curated/predictions_raw/public_samples_raw_curated.json'
cm_bio = '../../../resources/data/restricted/training-datasets/public/cm_train_candidates.txt'


In [ ]:
j10_candidates = generate_bio_from_train_candidates(j10_raw_samples, j10_bio)
pj_jujuy_candidates = generate_bio_from_train_candidates(pj_jujuy_raw_samples, pj_jujuy_bio)
saij_candidates = generate_bio_from_train_candidates(saij_raw_samples, saij_bio)
cm_candidates = generate_bio_from_train_candidates(cm_raw_samples, cm_bio)

In [ ]:
defensoria_raw_samples = '../../../resources/data/restricted/ner-langextract-alignment/defensoria/train_candidates.jsonl'
defensoria_bio = '../../../resources/data/restricted/training-datasets/private/defensoria_train_candidates.txt'

augmented_raw_samples = '../../../resources/data/public-data/augmented-paragraphs/train_candidates.jsonl'
augmented_bio = '../../../resources/data/restricted/training-datasets/public/augmented_train_candidates.txt'

In [ ]:
defensoria_candidates = generate_bio_from_train_candidates(defensoria_raw_samples, defensoria_bio)
augmented_candidates = generate_bio_from_train_candidates(augmented_raw_samples, augmented_bio)

# BIO Dataset Composition Report

Now that we have all the BIO txt's done we continue with the analysis.

This section analyzes BIO txt datasets in small reusable steps. It compares each J10 split independently (`train`, `dev`, `test`) and then reuses the same functions for every BIO txt under `resources/data/restricted/training-datasets`.


In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


BIO_ROOT = Path("../../../resources/data/restricted/training-datasets")
J10_OLD_DIR = BIO_ROOT / "private" / "J10-old-data"
J10_NEW_DIR = BIO_ROOT / "private" / "J10-new-data"
SPLITS = ["train", "dev", "test"]


pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## Step 1 - BIO Parsing Helpers

These functions parse BIO txt files into paragraphs and collect composition statistics for any single file or group of files.


In [ ]:
def label_from_bio_tag(tag: str) -> str | None:
    if tag == "O":
        return None
    if tag.startswith(("B-", "I-")):
        return tag[2:]
    return tag


def parse_bio_file(path: str | Path) -> list[list[tuple[str, str]]]:
    """Parse a BIO txt file into paragraphs of (token, tag)."""
    path = Path(path)
    paragraphs = []
    current = []

    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = line.strip()
        if not line:
            if current:
                paragraphs.append(current)
                current = []
            continue

        parts = line.rsplit(maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Invalid BIO line at {path}:{line_no}: {line!r}")
        current.append((parts[0], parts[1]))

    if current:
        paragraphs.append(current)
    return paragraphs


def count_entities(paragraph: list[tuple[str, str]]) -> Counter:
    """Count entity spans from BIO tags. Boundary I-* tags are counted as new spans."""
    counts = Counter()
    previous_label = None

    for _, tag in paragraph:
        label = label_from_bio_tag(tag)
        if label is None:
            previous_label = None
            continue

        if tag.startswith("B-") or label != previous_label:
            counts[label] += 1
        previous_label = label

    return counts


def collect_bio_stats_from_paragraphs(paragraphs: list[list[tuple[str, str]]]) -> dict[str, Any]:
    label_token_counts = Counter()
    entity_counts = Counter()
    tag_counts = Counter()
    paragraph_lengths = []
    labeled_paragraph_lengths = []

    for paragraph in paragraphs:
        paragraph_lengths.append(len(paragraph))
        paragraph_entity_counts = count_entities(paragraph)
        entity_counts.update(paragraph_entity_counts)
        has_label = bool(paragraph_entity_counts)

        if has_label:
            labeled_paragraph_lengths.append(len(paragraph))

        for _, tag in paragraph:
            tag_counts[tag] += 1
            label = label_from_bio_tag(tag)
            if label is not None:
                label_token_counts[label] += 1

    tokens = sum(paragraph_lengths)
    labeled_tokens = sum(label_token_counts.values())
    labeled_paragraphs = len(labeled_paragraph_lengths)

    return {
        "paragraphs": len(paragraphs),
        "labeled_paragraphs": labeled_paragraphs,
        "unlabeled_paragraphs": len(paragraphs) - labeled_paragraphs,
        "tokens": tokens,
        "labeled_tokens": labeled_tokens,
        "entity_mentions": sum(entity_counts.values()),
        "labels": sorted(entity_counts),
        "n_labels": len(entity_counts),
        "label_token_counts": label_token_counts,
        "entity_counts": entity_counts,
        "tag_counts": tag_counts,
        "paragraph_lengths": paragraph_lengths,
        "labeled_paragraph_lengths": labeled_paragraph_lengths,
        "labeled_paragraph_rate": labeled_paragraphs / max(1, len(paragraphs)),
        "labeled_token_rate": labeled_tokens / max(1, tokens),
    }


def collect_bio_file_stats(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    stats = collect_bio_stats_from_paragraphs(parse_bio_file(path))
    stats["path"] = path
    stats["dataset"] = path.parent.name
    stats["split"] = path.stem
    return stats


def collect_bio_group_stats(paths: list[Path]) -> dict[str, Any]:
    group_stats = collect_bio_stats_from_paragraphs([
        paragraph
        for path in paths
        for paragraph in parse_bio_file(path)
    ])
    group_stats["paths"] = paths
    return group_stats


def paragraph_token_key(paragraph: list[tuple[str, str]]) -> str:
    return " ".join(token for token, _ in paragraph)


def bio_paragraph_to_text(paragraph: list[tuple[str, str]]) -> str:
    return " ".join(f"{token} {tag}" for token, tag in paragraph)


## Step 2 - Display Helpers

These helpers convert stats into compact tables and histograms. Entity counts measure labeled spans; token counts measure every `B-*`/`I-*` token.


In [ ]:
def pct(value: float) -> str:
    return f"{value * 100:.1f}%"


def stats_summary_row(name: str, stats: dict[str, Any]) -> dict[str, Any]:
    return {
        "dataset": name,
        "paragraphs": stats["paragraphs"],
        "labeled_paragraphs": stats["labeled_paragraphs"],
        "unlabeled_paragraphs": stats["unlabeled_paragraphs"],
        "labeled_paragraph_rate": pct(stats["labeled_paragraph_rate"]),
        "tokens": stats["tokens"],
        "labeled_tokens": stats["labeled_tokens"],
        "labeled_token_rate": pct(stats["labeled_token_rate"]),
        "entity_mentions": stats["entity_mentions"],
        "labels": stats["n_labels"],
    }


def comparison_summary(old_name: str, old_stats: dict[str, Any], new_name: str, new_stats: dict[str, Any]) -> pd.DataFrame:
    rows = [
        stats_summary_row(old_name, old_stats),
        stats_summary_row(new_name, new_stats),
    ]
    delta = {
        "dataset": "delta new-old",
        "paragraphs": new_stats["paragraphs"] - old_stats["paragraphs"],
        "labeled_paragraphs": new_stats["labeled_paragraphs"] - old_stats["labeled_paragraphs"],
        "unlabeled_paragraphs": new_stats["unlabeled_paragraphs"] - old_stats["unlabeled_paragraphs"],
        "labeled_paragraph_rate": pct(new_stats["labeled_paragraph_rate"] - old_stats["labeled_paragraph_rate"]),
        "tokens": new_stats["tokens"] - old_stats["tokens"],
        "labeled_tokens": new_stats["labeled_tokens"] - old_stats["labeled_tokens"],
        "labeled_token_rate": pct(new_stats["labeled_token_rate"] - old_stats["labeled_token_rate"]),
        "entity_mentions": new_stats["entity_mentions"] - old_stats["entity_mentions"],
        "labels": new_stats["n_labels"] - old_stats["n_labels"],
    }
    return pd.DataFrame(rows + [delta])


def label_comparison(old_stats: dict[str, Any], new_stats: dict[str, Any]) -> pd.DataFrame:
    labels = sorted(set(old_stats["entity_counts"]) | set(new_stats["entity_counts"]))
    rows = []
    for label in labels:
        old_mentions = old_stats["entity_counts"].get(label, 0)
        new_mentions = new_stats["entity_counts"].get(label, 0)
        old_tokens = old_stats["label_token_counts"].get(label, 0)
        new_tokens = new_stats["label_token_counts"].get(label, 0)
        rows.append({
            "label": label,
            "old_mentions": old_mentions,
            "new_mentions": new_mentions,
            "delta_mentions": new_mentions - old_mentions,
            "old_tokens": old_tokens,
            "new_tokens": new_tokens,
            "delta_tokens": new_tokens - old_tokens,
        })
    return pd.DataFrame(rows).sort_values(["delta_mentions", "new_mentions"], ascending=[False, False])


def dataset_composition_table(stats_by_name: dict[str, dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame([
        stats_summary_row(name, stats)
        for name, stats in stats_by_name.items()
    ]).sort_values(["dataset"])


def label_inventory_table(stats_by_name: dict[str, dict[str, Any]], single_dataset: bool = False) -> pd.DataFrame:
    rows = []
    for name, stats in stats_by_name.items():
        for label, mentions in stats["entity_counts"].most_common():
            rows.append({
                "dataset": name,
                "label": label,
                "entity_mentions": mentions,
                "labeled_tokens": stats["label_token_counts"].get(label, 0),
            })

    columns = ["dataset", "label", "entity_mentions", "labeled_tokens"]
    if not rows:
        return pd.DataFrame(columns=columns)

    if single_dataset:
        columns.remove("dataset")
        return pd.DataFrame(rows, columns=columns).sort_values(
            ["entity_mentions"],
            ascending=[False],
        )
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["dataset", "entity_mentions"],
        ascending=[True, False],
    )

def plot_label_histogram(
    old_stats: dict[str, Any],
    new_stats: dict[str, Any],
    title: str,
    top_n: int | None = 30,
) -> None:
    comparison = label_comparison(old_stats, new_stats)
    comparison["total_mentions"] = comparison["old_mentions"] + comparison["new_mentions"]
    plot_df = comparison.sort_values("total_mentions", ascending=False)
    if top_n is not None:
        plot_df = plot_df.head(top_n)

    ax = plot_df.set_index("label")[["old_mentions", "new_mentions"]].plot(
        kind="bar",
        figsize=(14, 5),
        width=0.82,
    )
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("Entity mentions")
    ax.tick_params(axis="x", rotation=60)
    ax.legend(["old", "new"])
    plt.tight_layout()


def plot_labeled_paragraph_histogram(stats_by_name: dict[str, dict[str, Any]], title: str) -> None:
    rows = []
    for name, stats in stats_by_name.items():
        rows.append({"dataset": name, "paragraph_type": "labeled", "paragraphs": stats["labeled_paragraphs"]})
        rows.append({"dataset": name, "paragraph_type": "unlabeled", "paragraphs": stats["unlabeled_paragraphs"]})

    plot_df = pd.DataFrame(rows)
    ax = plot_df.pivot(index="dataset", columns="paragraph_type", values="paragraphs").plot(
        kind="bar",
        stacked=True,
        figsize=(12, 4),
    )
    ax.set_title(title)
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Paragraphs")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()

## Step 3 - Load J10 Old/New Split Pairs

Each split is loaded independently so `train old` is compared only with `train new`, `dev old` only with `dev new`, and `test old` only with `test new`.


In [ ]:
j10_split_stats = {}

for split in SPLITS:
    old_path = J10_OLD_DIR / f"{split}.txt"
    new_path = J10_NEW_DIR / f"{split}.txt"
    j10_split_stats[split] = {
        "old": collect_bio_file_stats(old_path),
        "new": collect_bio_file_stats(new_path),
    }

pd.DataFrame([
    {
        "split": split,
        "old_path": values["old"]["path"],
        "new_path": values["new"]["path"],
    }
    for split, values in j10_split_stats.items()
])

## Step 4 - Train Old vs Train New


In [ ]:
train_old_stats = j10_split_stats["train"]["old"]
train_new_stats = j10_split_stats["train"]["new"]

comparison_summary("train old", train_old_stats, "train new", train_new_stats)

In [ ]:
train_label_comparison = label_comparison(train_old_stats, train_new_stats)
train_label_comparison

#### `Conclusion:` We didn't lose any type of label with the correction, and also we gain some in most of them. The new dataset keeps the old original train labels.

In [ ]:
plot_label_histogram(train_old_stats, train_new_stats, "J10 train: old vs new label histogram")

## Step 5 - Dev Old vs Dev New


In [ ]:
dev_old_stats = j10_split_stats["dev"]["old"]
dev_new_stats = j10_split_stats["dev"]["new"]

comparison_summary("dev old", dev_old_stats, "dev new", dev_new_stats)

In [ ]:
dev_label_comparison = label_comparison(dev_old_stats, dev_new_stats)
dev_label_comparison

In [ ]:
plot_label_histogram(dev_old_stats, dev_new_stats, "J10 dev: old vs new label histogram")

#### `Conclusion:` The dev dataset had some wrong labeled tokens. 

## Step 6 - Test Old vs Test New


In [ ]:
test_old_stats = j10_split_stats["test"]["old"]
test_new_stats = j10_split_stats["test"]["new"]

comparison_summary("test old", test_old_stats, "test new", test_new_stats)

In [ ]:
test_label_comparison = label_comparison(test_old_stats, test_new_stats)
test_label_comparison

In [ ]:
plot_label_histogram(test_old_stats, test_new_stats, "J10 test: old vs new label histogram")

#### `Conclusion:` There were also wrong labels in the old test dataset.

## Step 7 - J10 Split Overview

This stacked histogram shows how many labeled and unlabeled paragraphs each J10 split contains.


In [ ]:
j10_named_stats = {
    f"{split} {version}": stats
    for split, values in j10_split_stats.items()
    for version, stats in values.items()
}

dataset_composition_table(j10_named_stats)

In [ ]:
plot_labeled_paragraph_histogram(j10_named_stats, "J10 labeled vs unlabeled paragraphs by split")

## Step 8 - Individual BIO Dataset Reports

Each subsection analyzes one BIO txt independently and compares its label coverage against the base train dataset. Missing required labels are highlighted in the coverage table.


In [ ]:
all_bio_paths = sorted(BIO_ROOT.rglob("*.txt"))
available_bio_datasets = pd.DataFrame({
    "relative_path": [str(path.relative_to(BIO_ROOT)) for path in all_bio_paths],
    "absolute_path": all_bio_paths,
})

BASE_TRAIN_PATH = J10_OLD_DIR / "train.txt"
BASE_TRAIN_NAME = str(BASE_TRAIN_PATH.relative_to(BIO_ROOT))
base_train_stats = collect_bio_file_stats(BASE_TRAIN_PATH)
required_train_labels = sorted(base_train_stats["entity_counts"])
required_train_label_set = set(required_train_labels)

all_bio_stats = {
    str(path.relative_to(BIO_ROOT)): collect_bio_file_stats(path)
    for path in all_bio_paths
}

available_bio_datasets

In [ ]:
def required_label_coverage_table(dataset_name: str, stats: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for label in required_train_labels:
        entity_mentions = stats["entity_counts"].get(label, 0)
        rows.append({
            "required_label": label,
            "base_train_mentions": base_train_stats["entity_counts"].get(label, 0),
            "dataset_mentions": entity_mentions,
            "present_in_dataset": entity_mentions > 0,
        })
    return pd.DataFrame(rows)


def summarize_required_label_absence(dataset_name: str, coverage: pd.DataFrame) -> pd.DataFrame:
    missing_labels = coverage.loc[~coverage["present_in_dataset"], "required_label"].tolist()
    return pd.DataFrame([{
        "dataset": dataset_name,
        "required_labels_present": int(coverage["present_in_dataset"].sum()),
        "required_labels_missing": len(missing_labels),
        "missing_labels": ", ".join(missing_labels),
    }])


def highlight_missing_coverage(row: pd.Series) -> list[str]:
    if not row["present_in_dataset"]:
        return ["background-color: #ffd6d6; color: #8a0000; font-weight: 700"] * len(row)
    return [""] * len(row)


def plot_single_dataset_label_histogram(stats: dict[str, Any], title: str, top_n: int | None = 30) -> None:
    label_counts = pd.Series(stats["entity_counts"], name="entity_mentions").sort_values(ascending=False)
    if top_n is not None:
        label_counts = label_counts.head(top_n)

    ax = label_counts.plot(kind="bar", figsize=(14, 5), width=0.82)
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("Entity mentions")
    ax.tick_params(axis="x", rotation=60)
    plt.tight_layout()


def display_individual_bio_report(relative_path: str, show_plots: bool = True) -> dict[str, Any]:
    stats = all_bio_stats[relative_path]
    named_stats = {relative_path: stats}
    coverage = required_label_coverage_table(relative_path, stats)
    absence_summary = summarize_required_label_absence(relative_path, coverage)

    display(dataset_composition_table(named_stats))
    display(label_inventory_table(named_stats, single_dataset=True))
    display(absence_summary)
    display(coverage.style.apply(highlight_missing_coverage, axis=1))

    if show_plots:
        plot_single_dataset_label_histogram(stats, f"{relative_path}: label histogram")
        plot_labeled_paragraph_histogram(named_stats, f"{relative_path}: labeled vs unlabeled paragraphs")

    return {
        "stats": stats,
        "coverage": coverage,
        "absence_summary": absence_summary,
    }

## Step 8.A - Analyze ***dev.txt***

Path: `private/J10-new-data/dev.txt`

In [ ]:
report_private_j10_new_train_candidates_txt = display_individual_bio_report("private/J10-new-data/dev.txt")

`Conclusion:` This is curious because there are two labels that aren't present in the dev dataset: ***IP*** and ***NUM_CAJA_AHORRO***.

## Step 8.B - Analyze ***test.txt***

Path: `private/J10-new-data/test.txt`

In [ ]:
report_private_j10_new_train_candidates_txt = display_individual_bio_report("private/J10-new-data/test.txt")

`Conclusion:` This is curious because there are two labels that aren't present in the dev dataset: ***IP*** and ***NOMBRE_ARCHIVO***.

## Step 8.C - Analyze ***j10_new_train_candidates.txt***

Path: `private/j10_new_train_candidates.txt`

In [ ]:
report_private_j10_new_train_candidates_txt = display_individual_bio_report("private/j10_new_train_candidates.txt")

## Step 8.D - Analyze ***defensoria_train_candidates.txt***

Path: `private/defensoria_train_candidates.txt`

In [ ]:
report_private_defensoria_train_candidates_txt = display_individual_bio_report("private/defensoria_train_candidates.txt")

## Step 8.E - Analyze ***pj_jujuy_train_candidates.txt***

Path: `private/pj_jujuy_train_candidates.txt`

In [ ]:
report_private_pj_jujuy_train_candidates_txt = display_individual_bio_report("private/pj_jujuy_train_candidates.txt")

## Step 8.F - Analyze ***saij_train_candidates.txt***

Path: `private/saij_train_candidates.txt`

In [ ]:
report_private_saij_train_candidates_txt = display_individual_bio_report("private/saij_train_candidates.txt")

## Step 8.G - Analyze ***augmented_train_candidates.txt***

Path: `public/augmented_train_candidates.txt`

In [ ]:
report_public_augmented_train_candidates_txt = display_individual_bio_report("public/augmented_train_candidates.txt")

## Step 8.H - Analyze ***cm_train_candidates.txt***

Path: `public/cm_train_candidates.txt`

In [ ]:
report_public_cm_train_candidates_txt = display_individual_bio_report("public/cm_train_candidates.txt")

## Step 8.I - Analyze ***pii_train_candidates.txt***

Path: `public/pii_train_candidates.txt`

In [ ]:
report_public_pii_train_candidates_txt = display_individual_bio_report("public/pii_train_candidates.txt")

## Step 8.J - Analyze ***synthetic_train_candidates.txt***

Path: `public/synthetic_train_candidates.txt`

In [ ]:
report_public_synthetic_train_candidates_txt = display_individual_bio_report("public/synthetic_train_candidates.txt")

## Step 9 - Required Train Label Coverage Matrix

This section summarizes base-train label coverage only for candidate datasets outside the original/reviewed J10 split folders. It excludes every file under `private/J10-old-data` and `private/J10-new-data`, while still using `private/J10-old-data/train.txt` as the required label baseline.


In [ ]:
BASE_TRAIN_PATH = J10_OLD_DIR / "train.txt"
BASE_TRAIN_NAME = str(BASE_TRAIN_PATH.relative_to(BIO_ROOT))

base_train_stats = collect_bio_file_stats(BASE_TRAIN_PATH)
required_train_labels = sorted(base_train_stats["entity_counts"])
required_train_label_set = set(required_train_labels)

required_label_table = pd.DataFrame({
    "required_label": required_train_labels,
    "base_train_mentions": [base_train_stats["entity_counts"][label] for label in required_train_labels],
    "base_train_labeled_tokens": [base_train_stats["label_token_counts"][label] for label in required_train_labels],
})

required_label_table

In [ ]:
if "all_bio_paths" not in globals():
    all_bio_paths = sorted(BIO_ROOT.rglob("*.txt"))

EXCLUDED_COMBINATION_DIRS = [
    J10_OLD_DIR,
    J10_NEW_DIR,
]


def is_training_combination_candidate(path: Path) -> bool:
    path = Path(path)
    return not any(path.is_relative_to(excluded_dir) for excluded_dir in EXCLUDED_COMBINATION_DIRS)


coverage_candidate_paths = [
    path
    for path in all_bio_paths
    if is_training_combination_candidate(path)
]

coverage_candidate_stats = {
    str(path.relative_to(BIO_ROOT)): collect_bio_file_stats(path)
    for path in coverage_candidate_paths
}

candidate_dataset_table = pd.DataFrame({
    "candidate_dataset": sorted(coverage_candidate_stats),
})

display(candidate_dataset_table)

label_presence_rows = []
for dataset_name, stats in coverage_candidate_stats.items():
    row = {"dataset": dataset_name}
    row.update({label: stats["entity_counts"].get(label, 0) for label in required_train_labels})
    label_presence_rows.append(row)

label_presence_matrix = pd.DataFrame(label_presence_rows).set_index("dataset")
label_presence_matrix


In [ ]:
def highlight_missing_required_labels(value: int) -> str:
    if value == 0:
        return "background-color: #ffd6d6; color: #8a0000; font-weight: 700"
    return ""


label_presence_matrix_styled = label_presence_matrix.style.map(highlight_missing_required_labels)
label_presence_matrix_styled


In [ ]:
label_absence_summary = pd.DataFrame([
    {
        "dataset": dataset_name,
        "required_labels_present": int((row > 0).sum()),
        "required_labels_missing": int((row == 0).sum()),
        "missing_labels": ", ".join(row.index[row == 0]),
    }
    for dataset_name, row in label_presence_matrix.iterrows()
]).sort_values(["required_labels_missing", "dataset"])

label_absence_summary


In [ ]:
label_missing_by_dataset_count = pd.DataFrame({
    "required_label": required_train_labels,
    "base_train_mentions": [base_train_stats["entity_counts"][label] for label in required_train_labels],
    "datasets_missing_label": [(label_presence_matrix[label] == 0).sum() for label in required_train_labels],
    "datasets_with_label": [(label_presence_matrix[label] > 0).sum() for label in required_train_labels],
}).sort_values(["datasets_missing_label", "base_train_mentions"], ascending=[False, True])

label_missing_by_dataset_count


## Step 9B - Dataset Combination Coverage

This checks combinations only across `coverage_candidate_stats`, which excludes `private/J10-old-data` and `private/J10-new-data`. The required labels still come from `BASE_TRAIN_PATH`.


In [ ]:
def missing_required_labels(stats: dict[str, Any], required_labels: set[str]) -> list[str]:
    return sorted(required_labels - set(stats["entity_counts"]))


def combine_stats(names: list[str], stats_by_name: dict[str, dict[str, Any]]) -> dict[str, Any]:
    label_token_counts = Counter()
    entity_counts = Counter()
    tag_counts = Counter()
    paragraph_lengths = []
    labeled_paragraph_lengths = []

    for name in names:
        stats = stats_by_name[name]
        label_token_counts.update(stats["label_token_counts"])
        entity_counts.update(stats["entity_counts"])
        tag_counts.update(stats["tag_counts"])
        paragraph_lengths.extend(stats["paragraph_lengths"])
        labeled_paragraph_lengths.extend(stats["labeled_paragraph_lengths"])

    tokens = sum(paragraph_lengths)
    labeled_tokens = sum(label_token_counts.values())
    labeled_paragraphs = len(labeled_paragraph_lengths)

    return {
        "names": names,
        "paragraphs": len(paragraph_lengths),
        "labeled_paragraphs": labeled_paragraphs,
        "unlabeled_paragraphs": len(paragraph_lengths) - labeled_paragraphs,
        "tokens": tokens,
        "labeled_tokens": labeled_tokens,
        "entity_mentions": sum(entity_counts.values()),
        "labels": sorted(entity_counts),
        "n_labels": len(entity_counts),
        "label_token_counts": label_token_counts,
        "entity_counts": entity_counts,
        "tag_counts": tag_counts,
        "paragraph_lengths": paragraph_lengths,
        "labeled_paragraph_lengths": labeled_paragraph_lengths,
        "labeled_paragraph_rate": labeled_paragraphs / max(1, len(paragraph_lengths)),
        "labeled_token_rate": labeled_tokens / max(1, tokens),
    }


def evaluate_dataset_combinations(
    stats_by_name: dict[str, dict[str, Any]],
    required_labels: set[str],
    max_combination_size: int = 3,
) -> pd.DataFrame:
    rows = []
    names = sorted(stats_by_name)
    for size in range(1, max_combination_size + 1):
        for combo in combinations(names, size):
            combo_stats = combine_stats(list(combo), stats_by_name)
            missing = missing_required_labels(combo_stats, required_labels)
            rows.append({
                "datasets": " + ".join(combo),
                "n_datasets": size,
                "paragraphs": combo_stats["paragraphs"],
                "labeled_paragraphs": combo_stats["labeled_paragraphs"],
                "entity_mentions": combo_stats["entity_mentions"],
                "labels": combo_stats["n_labels"],
                "required_labels_present": len(required_labels) - len(missing),
                "required_labels_missing": len(missing),
                "missing_labels": ", ".join(missing),
                "passes_train_label_floor": len(missing) == 0,
            })
    return pd.DataFrame(rows).sort_values(
        ["passes_train_label_floor", "required_labels_missing", "labeled_paragraphs"],
        ascending=[False, True, False],
    )


combination_report = evaluate_dataset_combinations(
    coverage_candidate_stats,
    required_train_label_set,
    max_combination_size=3,
)
combination_report

In [ ]:
valid_training_combinations = combination_report[
    combination_report["passes_train_label_floor"]
]

valid_training_combinations.head(20)

## Step 10 - New Dataset Aggregate Stats

This section summarizes only the new/external candidate datasets used for combinations. It excludes every file under `private/J10-old-data` and `private/J10-new-data`.


In [ ]:
if "coverage_candidate_stats" not in globals():
    coverage_candidate_paths = [
        path
        for path in all_bio_paths
        if is_training_combination_candidate(path)
    ]
    coverage_candidate_stats = {
        str(path.relative_to(BIO_ROOT)): collect_bio_file_stats(path)
        for path in coverage_candidate_paths
    }

new_dataset_total_stats = combine_stats(
    sorted(coverage_candidate_stats),
    coverage_candidate_stats,
)

new_dataset_totals = pd.DataFrame([{
    "datasets": len(coverage_candidate_stats),
    "paragraphs": new_dataset_total_stats["paragraphs"],
    "labeled_paragraphs": new_dataset_total_stats["labeled_paragraphs"],
    "unlabeled_paragraphs": new_dataset_total_stats["unlabeled_paragraphs"],
    "labeled_paragraph_rate": pct(new_dataset_total_stats["labeled_paragraph_rate"]),
    "tokens": new_dataset_total_stats["tokens"],
    "labeled_tokens": new_dataset_total_stats["labeled_tokens"],
    "entity_mentions": new_dataset_total_stats["entity_mentions"],
    "unique_labels": new_dataset_total_stats["n_labels"],
}])

new_dataset_totals


In [ ]:
new_dataset_composition_by_file = dataset_composition_table(coverage_candidate_stats)
new_dataset_total_row = pd.DataFrame([
    stats_summary_row("TOTAL new datasets", new_dataset_total_stats)
])

new_dataset_composition_with_total = pd.concat(
    [new_dataset_composition_by_file, new_dataset_total_row],
    ignore_index=True,
)

new_dataset_composition_with_total


In [ ]:
new_dataset_label_composition_total = pd.DataFrame([
    {
        "label": label,
        "entity_mentions": new_dataset_total_stats["entity_counts"].get(label, 0),
        "labeled_tokens": new_dataset_total_stats["label_token_counts"].get(label, 0),
        "datasets_with_label": sum(
            1 for stats in coverage_candidate_stats.values()
            if stats["entity_counts"].get(label, 0) > 0
        ),
        "datasets_missing_label": sum(
            1 for stats in coverage_candidate_stats.values()
            if stats["entity_counts"].get(label, 0) == 0
        ),
    }
    for label in sorted(new_dataset_total_stats["entity_counts"])
]).sort_values(["entity_mentions", "label"], ascending=[False, True])

new_dataset_label_composition_total


In [ ]:
new_dataset_label_composition_by_file = label_inventory_table(coverage_candidate_stats)
new_dataset_label_composition_by_file

In [ ]:
plot_labeled_paragraph_histogram(
    coverage_candidate_stats,
    "New/external datasets: labeled vs unlabeled paragraphs",
)

In [ ]:

plot_single_dataset_label_histogram(
    new_dataset_total_stats,
    "New/external datasets: total label composition",
    top_n=None,
)
